In [1]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)

Project Root: /home/ayon/git/EventCameraProject


In [4]:
import torch

from src.models.world_model.temporal_encoder import TemporalEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"

model = TemporalEncoder(
    input_channels=128,
    hidden_channels=128,
).to(device)

x = torch.randn(
    2,
    4,
    128,
    60,
    80,
    device=device,
    requires_grad=True,
)

y = model(x)

print("Input :", x.shape)
print("Output:", y.shape)

assert y.shape == (2, 4, 128, 60, 80)

loss = y.mean()
loss.backward()

print("✓ Gradient OK")

params = sum(p.numel() for p in model.parameters())

print(f"Parameters: {params:,}")

Input : torch.Size([2, 4, 128, 60, 80])
Output: torch.Size([2, 4, 128, 60, 80])
✓ Gradient OK
Parameters: 1,180,160


In [5]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)

# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------

import torch
from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

# ---------------------------------------------------------------------
# Device
# ---------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# ---------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------

dataset_root = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

frame_dataset = EVIMO2Dataset(
    dataset_root=dataset_root,
    sensors=("left_camera", "right_camera"),
    split="train",
    load_depth=True,
    load_mask=True,
)

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=(-3, -2, -1, 0),
)

loader = DataLoader(
    temporal_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=temporal_collate_fn,
)

# ---------------------------------------------------------------------
# Transform Pipeline
# ---------------------------------------------------------------------

transform = Compose(
    [
        ToTensor(),
        NormalizeEventTime(),
        NormalizeIMU(),
        VoxelizeEvents(
            num_bins=5,
        ),
    ]
)

# ---------------------------------------------------------------------
# Get one batch
# ---------------------------------------------------------------------

raw_batch = next(iter(loader))

voxel_batch = transform(raw_batch)

# ---------------------------------------------------------------------
# Build Event Tensor
# ---------------------------------------------------------------------

voxels = torch.stack(
    [
        frame.voxel_grid
        for frame in voxel_batch.frames
    ],
    dim=1,
).to(device)

print()
print("=" * 90)
print("EVENT VOXELS")
print("=" * 90)
print("Shape :", tuple(voxels.shape))
print("dtype :", voxels.dtype)
print("device:", voxels.device)

assert voxels.ndim == 5

batch_size, sequence_length, num_bins, height, width = voxels.shape

print()
print(f"Batch Size      : {batch_size}")
print(f"Sequence Length : {sequence_length}")
print(f"Voxel Bins      : {num_bins}")
print(f"Resolution      : {height} x {width}")

Project Root: /home/ayon/git/EventCameraProject
Device: cpu
EVIMO2 Sequence Index
Sequences : 22
Frames    : 9352
Sensors   : left_camera, right_camera
Split     : train

EVENT VOXELS
Shape : (2, 4, 5, 480, 640)
dtype : torch.float32
device: cpu

Batch Size      : 2
Sequence Length : 4
Voxel Bins      : 5
Resolution      : 480 x 640


In [6]:
print("=" * 80)
print("VOXEL CAMERA TEST")
print("=" * 80)

frame = voxel_batch.frames[-1]

print()
print("Voxel")
print(frame.voxel_grid.shape)

print()
print("Intrinsics")
print(frame.camera_intrinsics.shape)
print(frame.camera_intrinsics[0])

print()
print("Distortion")
print(frame.camera_distortion.shape)
print(frame.camera_distortion[0])

VOXEL CAMERA TEST

Voxel
torch.Size([2, 5, 480, 640])

Intrinsics
torch.Size([2, 3, 3])
tensor([[556.5740,   0.0000, 321.8460],
        [  0.0000, 556.0870, 225.6180],
        [  0.0000,   0.0000,   1.0000]])

Distortion
torch.Size([2, 4])
tensor([-0.1083,  0.2062,  0.0000,  0.0000])


In [7]:
raw_batch = next(iter(loader))
frame = raw_batch.frames[0]


print("="*80)
print("COLLATE CAMERA TEST")
print("="*80)


print(
    "Number of samples:",
    len(frame.camera_intrinsics)
)


for i in range(len(frame.camera_intrinsics)):

    print()
    print("Sample:", i)

    print("K:")
    print(frame.camera_intrinsics[i])

    print()

    print("D:")
    print(frame.camera_distortion[i])

COLLATE CAMERA TEST
Number of samples: 2

Sample: 0
K:
[[556.574   0.    321.846]
 [  0.    556.087 225.618]
 [  0.      0.      1.   ]]

D:
[-0.10828   0.206233  0.        0.      ]

Sample: 1
K:
[[554.585   0.    327.556]
 [  0.    554.21  202.321]
 [  0.      0.      1.   ]]

D:
[-0.094841  0.187073  0.        0.      ]


In [8]:
for t, frame in enumerate(voxel_batch.frames):

    print(f"\nFrame {t}")

    print("Voxel :", tuple(frame.voxel_grid.shape))
    print("IMU timestamps :", tuple(frame.imu_timestamps.shape))
    print("Gyro :", tuple(frame.imu_angular_velocity.shape))
    print("Accel :", tuple(frame.imu_linear_acceleration.shape))


Frame 0
Voxel : (2, 5, 480, 640)
IMU timestamps : (34,)
Gyro : (34, 3)
Accel : (34, 3)

Frame 1
Voxel : (2, 5, 480, 640)
IMU timestamps : (34,)
Gyro : (34, 3)
Accel : (34, 3)

Frame 2
Voxel : (2, 5, 480, 640)
IMU timestamps : (34,)
Gyro : (34, 3)
Accel : (34, 3)

Frame 3
Voxel : (2, 5, 480, 640)
IMU timestamps : (34,)
Gyro : (34, 3)
Accel : (34, 3)


In [9]:
from src.models.world_model.event_encoder import EventEncoder

# ==========================================================
# Event Encoder Test
# ==========================================================

print()
print("=" * 90)
print("EVENT ENCODER TEST")
print("=" * 90)

encoder = EventEncoder(
    input_channels=num_bins,
).to(device)

encoder.train()

# ----------------------------------------------------------
# Forward
# ----------------------------------------------------------

features = encoder(voxels)

assert isinstance(features, list)
assert len(features) == 4

expected_channels = [32, 64, 128, 256]
expected_sizes = [
    (240, 320),
    (120, 160),
    (60, 80),
    (30, 40),
]

print()

print("Feature Pyramid")
print("-" * 90)

for level, feature in enumerate(features):

    print(f"\nLevel {level + 1}")

    print("Shape :", tuple(feature.shape))
    print("dtype :", feature.dtype)
    print("device:", feature.device)

    b, t, c, h, w = feature.shape

    assert b == batch_size
    assert t == sequence_length

    assert c == expected_channels[level]
    assert (h, w) == expected_sizes[level]

    #
    # Numerical sanity
    #

    assert torch.isfinite(feature).all()

    print(f"Min  : {feature.min().item():.5f}")
    print(f"Max  : {feature.max().item():.5f}")
    print(f"Mean : {feature.mean().item():.5f}")
    print(f"Std  : {feature.std().item():.5f}")

# ----------------------------------------------------------
# Gradient Check
# ----------------------------------------------------------

loss = sum(
    feature.mean()
    for feature in features
)

encoder.zero_grad()

loss.backward()

print()
print("-" * 90)
print("Gradient Check")
print("-" * 90)

num_grad = 0

for name, param in encoder.named_parameters():

    if param.grad is None:
        continue

    assert torch.isfinite(param.grad).all()

    num_grad += 1

print(f"Parameters with gradients : {num_grad}")

assert num_grad > 0

# ----------------------------------------------------------
# GPU Memory
# ----------------------------------------------------------

if torch.cuda.is_available():

    print()
    print("-" * 90)
    print("CUDA")
    print("-" * 90)

    print(
        f"Allocated : "
        f"{torch.cuda.memory_allocated()/1024**2:.2f} MB"
    )

    print(
        f"Reserved  : "
        f"{torch.cuda.memory_reserved()/1024**2:.2f} MB"
    )

print()
print("=" * 90)
print("✓ EVENT ENCODER TEST PASSED")
print("=" * 90)


EVENT ENCODER TEST

Feature Pyramid
------------------------------------------------------------------------------------------

Level 1
Shape : (2, 4, 32, 240, 320)
dtype : torch.float32
device: cpu
Min  : -0.27846
Max  : 45.71085
Mean : 0.32832
Std  : 1.24121

Level 2
Shape : (2, 4, 64, 120, 160)
dtype : torch.float32
device: cpu
Min  : -0.27846
Max  : 38.19544
Mean : 0.31568
Std  : 1.08974

Level 3
Shape : (2, 4, 128, 60, 80)
dtype : torch.float32
device: cpu
Min  : -0.27846
Max  : 29.05613
Mean : 0.34841
Std  : 1.10488

Level 4
Shape : (2, 4, 256, 30, 40)
dtype : torch.float32
device: cpu
Min  : -0.27846
Max  : 24.31401
Mean : 0.35377
Std  : 1.10806

------------------------------------------------------------------------------------------
Gradient Check
------------------------------------------------------------------------------------------
Parameters with gradients : 66

✓ EVENT ENCODER TEST PASSED


In [10]:
frame = voxel_batch.frames[0]

print(frame.imu_sample_indices)

print(torch.unique(frame.imu_sample_indices))
print(torch.bincount(frame.imu_sample_indices))

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
tensor([0, 1])
tensor([17, 17])


In [11]:
from src.models.world_model.imu_encoder import IMUEncoder

# ==========================================================
# IMU ENCODER TEST
# ==========================================================

print()
print("=" * 90)
print("IMU ENCODER TEST")
print("=" * 90)

imu_encoder = IMUEncoder(
    hidden_channels=64,
    embedding_dim=128,
).to(device)

imu_encoder.train()

# ----------------------------------------------------------
# Forward
# ----------------------------------------------------------

motion_embeddings = []

for temporal_index, frame in enumerate(voxel_batch.frames):

    embedding = imu_encoder(
        frame=frame,
        batch_size=batch_size,
    )

    motion_embeddings.append(embedding)

motion_embeddings = torch.stack(
    motion_embeddings,
    dim=1,
)

# ----------------------------------------------------------
# Shape
# ----------------------------------------------------------

print()

print("Motion Embedding")
print("-" * 90)

print("Shape :", tuple(motion_embeddings.shape))
print("dtype :", motion_embeddings.dtype)
print("device:", motion_embeddings.device)

assert motion_embeddings.ndim == 3

assert motion_embeddings.shape == (
    batch_size,
    sequence_length,
    128,
)

# ----------------------------------------------------------
# Statistics
# ----------------------------------------------------------

print()

print("Statistics")

print(f"Min  : {motion_embeddings.min().item():.5f}")
print(f"Max  : {motion_embeddings.max().item():.5f}")
print(f"Mean : {motion_embeddings.mean().item():.5f}")
print(f"Std  : {motion_embeddings.std().item():.5f}")

assert torch.isfinite(
    motion_embeddings
).all()

# ----------------------------------------------------------
# Temporal Consistency
# ----------------------------------------------------------

print()

print("-" * 90)
print("Temporal Consistency")
print("-" * 90)

for t in range(sequence_length):

    mean_feature = (
        motion_embeddings[:, t]
        .abs()
        .mean()
        .item()
    )

    print(
        f"Frame {t}: {mean_feature:.5f}"
    )

# ----------------------------------------------------------
# Gradient Check
# ----------------------------------------------------------

loss = motion_embeddings.mean()

imu_encoder.zero_grad()

loss.backward()

print()

print("-" * 90)
print("Gradient Check")
print("-" * 90)

num_grad = 0

for name, parameter in imu_encoder.named_parameters():

    if parameter.grad is None:
        continue

    assert torch.isfinite(
        parameter.grad
    ).all()

    num_grad += 1

print(
    "Parameters with gradients:",
    num_grad,
)

assert num_grad > 0

# ----------------------------------------------------------
# GPU Memory
# ----------------------------------------------------------

if torch.cuda.is_available():

    print()

    print("-" * 90)
    print("CUDA")
    print("-" * 90)

    print(
        f"Allocated : "
        f"{torch.cuda.memory_allocated()/1024**2:.2f} MB"
    )

    print(
        f"Reserved  : "
        f"{torch.cuda.memory_reserved()/1024**2:.2f} MB"
    )

print()
print("=" * 90)
print("✓ IMU ENCODER TEST PASSED")
print("=" * 90)


IMU ENCODER TEST

Motion Embedding
------------------------------------------------------------------------------------------
Shape : (2, 4, 128)
dtype : torch.float32
device: cpu

Statistics
Min  : -0.88876
Max  : 0.88275
Mean : -0.02388
Std  : 0.38594

------------------------------------------------------------------------------------------
Temporal Consistency
------------------------------------------------------------------------------------------
Frame 0: 0.32271
Frame 1: 0.32373
Frame 2: 0.32485
Frame 3: 0.32524

------------------------------------------------------------------------------------------
Gradient Check
------------------------------------------------------------------------------------------
Parameters with gradients: 13

✓ IMU ENCODER TEST PASSED


In [12]:
from src.models.world_model.fusion import MotionFusion


print()
print("=" * 90)
print("MOTION FUSION TEST")
print("=" * 90)


# ----------------------------------------------------------
# Use outputs from previous tests
# ----------------------------------------------------------

event_feature = features[-1].detach()

imu_feature = motion_embeddings.detach()

print()
print("Inputs")
print("-" * 90)

print(
    "Event feature:",
    tuple(event_feature.shape)
)

print(
    "IMU feature:",
    tuple(imu_feature.shape)
)


# ----------------------------------------------------------
# Fusion
# ----------------------------------------------------------

fusion = MotionFusion(
    event_channels=256,
    imu_dim=128,
).to(device)


fusion.train()


fused = fusion(
    event_feature,
    imu_feature,
)


# ----------------------------------------------------------
# Check output
# ----------------------------------------------------------

print()

print("Output")
print("-" * 90)

print(
    "Fused:",
    tuple(fused.shape)
)


assert fused.shape == event_feature.shape


assert torch.isfinite(
    fused
).all()


print()

print("Statistics")

print(
    "Mean:",
    fused.mean().item()
)

print(
    "Std:",
    fused.std().item()
)


# ----------------------------------------------------------
# Gradient
# ----------------------------------------------------------

loss = fused.mean()

fusion.zero_grad()

loss.backward()


num_grad = 0

for name, param in fusion.named_parameters():

    if param.grad is not None:

        assert torch.isfinite(
            param.grad
        ).all()

        num_grad += 1


print()

print(
    "Parameters with gradients:",
    num_grad
)


assert num_grad > 0


print()
print("=" * 90)
print("✓ MOTION FUSION TEST PASSED")
print("=" * 90)


MOTION FUSION TEST

Inputs
------------------------------------------------------------------------------------------
Event feature: (2, 4, 256, 30, 40)
IMU feature: (2, 4, 128)

Output
------------------------------------------------------------------------------------------
Fused: (2, 4, 256, 30, 40)

Statistics
Mean: 0.14918066561222076
Std: 0.643358051776886

Parameters with gradients: 5

✓ MOTION FUSION TEST PASSED


In [13]:
# ==========================================================
# TEMPORAL ENCODER TEST
# ==========================================================

from src.models.world_model.temporal_encoder import TemporalEncoder


print()
print("=" * 90)
print("TEMPORAL ENCODER TEST")
print("=" * 90)


# ----------------------------------------------------------
# Input
# ----------------------------------------------------------

print()

print("Input")
print("-" * 90)

print(
    "Fused feature:",
    tuple(fused.shape)
)


# ----------------------------------------------------------
# Encoder
# ----------------------------------------------------------

temporal_encoder = TemporalEncoder(

    input_channels=256,

    hidden_channels=256,

    kernel_size=3,

).to(device)


temporal_encoder.train()


temporal_features = temporal_encoder(
    fused.detach()
)


# ----------------------------------------------------------
# Output
# ----------------------------------------------------------

print()

print("Output")
print("-" * 90)

print(
    "Temporal features:",
    tuple(temporal_features.shape)
)


assert temporal_features.shape == (
    fused.shape[0],
    fused.shape[1],
    256,
    fused.shape[3],
    fused.shape[4],
)


assert torch.isfinite(
    temporal_features
).all()


# ----------------------------------------------------------
# Statistics
# ----------------------------------------------------------

print()

print("Statistics")

print(
    "Mean:",
    temporal_features.mean().item()
)

print(
    "Std:",
    temporal_features.std().item()
)


# ----------------------------------------------------------
# Gradient Check
# ----------------------------------------------------------

loss = temporal_features.mean()


temporal_encoder.zero_grad()


loss.backward()


num_grad = 0


for name, parameter in temporal_encoder.named_parameters():

    if parameter.grad is not None:

        assert torch.isfinite(
            parameter.grad
        ).all()

        num_grad += 1


print()

print(
    "Parameters with gradients:",
    num_grad
)


assert num_grad > 0


print()

print("=" * 90)
print("✓ TEMPORAL ENCODER TEST PASSED")
print("=" * 90)


TEMPORAL ENCODER TEST

Input
------------------------------------------------------------------------------------------
Fused feature: (2, 4, 256, 30, 40)

Output
------------------------------------------------------------------------------------------
Temporal features: (2, 4, 256, 30, 40)

Statistics
Mean: -0.0006366248708218336
Std: 0.07348016649484634

Parameters with gradients: 2

✓ TEMPORAL ENCODER TEST PASSED


In [14]:
# ==========================================================
# WORLD TRANSITION TEST
# ==========================================================

from src.models.world_model.world_transition import WorldTransition


print()
print("=" * 90)
print("WORLD TRANSITION TEST")
print("=" * 90)


# ----------------------------------------------------------
# Prepare inputs
# ----------------------------------------------------------

previous_state = temporal_features[:, -2].detach()

target_state = temporal_features[:, -1].detach()

motion = motion_embeddings[:, -1].detach()


print()
print("Inputs")
print("-" * 90)


print(
    "Previous state:",
    tuple(previous_state.shape)
)

print(
    "Target state:",
    tuple(target_state.shape)
)

print(
    "Motion:",
    tuple(motion.shape)
)


# ----------------------------------------------------------
# Build transition model
# ----------------------------------------------------------

transition = WorldTransition(
    state_channels=256,
    motion_dim=128,
).to(device)


transition.train()


# ----------------------------------------------------------
# Forward
# ----------------------------------------------------------

predicted_state = transition(
    previous_state,
    motion,
)


# ----------------------------------------------------------
# Output check
# ----------------------------------------------------------

print()

print("Output")
print("-" * 90)


print(
    "Predicted state:",
    tuple(predicted_state.shape)
)


assert predicted_state.shape == previous_state.shape


assert torch.isfinite(
    predicted_state
).all()


# ----------------------------------------------------------
# Statistics
# ----------------------------------------------------------

print()

print("Statistics")

print(
    "Mean:",
    predicted_state.mean().item()
)

print(
    "Std:",
    predicted_state.std().item()
)


# ----------------------------------------------------------
# Initial prediction error
# ----------------------------------------------------------

with torch.no_grad():

    error = (
        predicted_state - target_state
    ).abs().mean()


print()

print(
    "Initial latent prediction error:",
    error.item()
)


# ----------------------------------------------------------
# Gradient Check
# ----------------------------------------------------------

loss = predicted_state.mean()


transition.zero_grad()


loss.backward()


num_grad = 0


for name, parameter in transition.named_parameters():

    if parameter.grad is not None:

        assert torch.isfinite(
            parameter.grad
        ).all()

        num_grad += 1


print()

print(
    "Parameters with gradients:",
    num_grad
)


assert num_grad > 0


print()

print("=" * 90)
print("✓ WORLD TRANSITION TEST PASSED")
print("=" * 90)


WORLD TRANSITION TEST

Inputs
------------------------------------------------------------------------------------------
Previous state: (2, 256, 30, 40)
Target state: (2, 256, 30, 40)
Motion: (2, 128)

Output
------------------------------------------------------------------------------------------
Predicted state: (2, 256, 30, 40)

Statistics
Mean: 0.20883765816688538
Std: 0.5593059659004211

Initial latent prediction error: 0.4119742214679718

Parameters with gradients: 8

✓ WORLD TRANSITION TEST PASSED


In [15]:
# ==========================================================
# LATENT RESIDUAL TEST
# ==========================================================

from src.models.world_model.residual import LatentResidual


print()
print("=" * 90)
print("LATENT RESIDUAL TEST")
print("=" * 90)


# ----------------------------------------------------------
# Inputs from previous test
# ----------------------------------------------------------

predicted = predicted_state.detach()

observed = target_state.detach()


print()

print("Inputs")
print("-" * 90)

print(
    "Predicted:",
    tuple(predicted.shape)
)

print(
    "Observed:",
    tuple(observed.shape)
)



# ----------------------------------------------------------
# Residual
# ----------------------------------------------------------

residual_module = LatentResidual().to(device)


residual = residual_module(
    predicted,
    observed,
)


# ----------------------------------------------------------
# Check
# ----------------------------------------------------------

print()

print("Output")
print("-" * 90)

print(
    "Residual:",
    tuple(residual.shape)
)


assert residual.shape == predicted.shape


assert torch.isfinite(
    residual
).all()


# ----------------------------------------------------------
# Statistics
# ----------------------------------------------------------

print()

print("Statistics")

print(
    "Min:",
    residual.min().item()
)

print(
    "Max:",
    residual.max().item()
)

print(
    "Mean:",
    residual.mean().item()
)

print(
    "Std:",
    residual.std().item()
)


# ----------------------------------------------------------
# Zero test
# ----------------------------------------------------------

zero_residual = residual_module(
    predicted,
    predicted,
)


assert torch.allclose(
    zero_residual,
    torch.zeros_like(zero_residual)
)


print()

print("=" * 90)
print("✓ LATENT RESIDUAL TEST PASSED")
print("=" * 90)


LATENT RESIDUAL TEST

Inputs
------------------------------------------------------------------------------------------
Predicted: (2, 256, 30, 40)
Observed: (2, 256, 30, 40)

Output
------------------------------------------------------------------------------------------
Residual: (2, 256, 30, 40)

Statistics
Min: 1.695007085800171e-07
Max: 6.864894390106201
Mean: 0.4119742214679718
Std: 0.43499302864074707

✓ LATENT RESIDUAL TEST PASSED


In [16]:
# ==========================================================
# DYNAMIC HEAD TEST
# ==========================================================

from src.models.world_model.dynamic_head import DynamicHead


print()
print("=" * 90)
print("DYNAMIC HEAD TEST")
print("=" * 90)


# ----------------------------------------------------------
# Input
# ----------------------------------------------------------

dynamic_head = DynamicHead(
    input_channels=256,
).to(device)


dynamic_head.train()


print()

print("Input")
print("-" * 90)

print(
    "Residual:",
    tuple(residual.shape)
)


# ----------------------------------------------------------
# Forward
# ----------------------------------------------------------

dynamic_probability = dynamic_head(
    residual
)


# ----------------------------------------------------------
# Output
# ----------------------------------------------------------

print()

print("Output")
print("-" * 90)

print(
    "Dynamic probability:",
    tuple(dynamic_probability.shape)
)


assert dynamic_probability.shape == (
    residual.shape[0],
    1,
    residual.shape[2],
    residual.shape[3],
)


assert torch.isfinite(
    dynamic_probability
).all()



# ----------------------------------------------------------
# Range check
# ----------------------------------------------------------

print()

print("Range")

print(
    "Min:",
    dynamic_probability.min().item()
)

print(
    "Max:",
    dynamic_probability.max().item()
)


assert dynamic_probability.min() >= 0

assert dynamic_probability.max() <= 1



# ----------------------------------------------------------
# Gradient
# ----------------------------------------------------------

loss = dynamic_probability.mean()


dynamic_head.zero_grad()


loss.backward()


num_grad = 0


for name, param in dynamic_head.named_parameters():

    if param.grad is not None:

        assert torch.isfinite(
            param.grad
        ).all()

        num_grad += 1



print()

print(
    "Parameters with gradients:",
    num_grad
)


assert num_grad > 0



print()

print("=" * 90)
print("✓ DYNAMIC HEAD TEST PASSED")
print("=" * 90)


DYNAMIC HEAD TEST

Input
------------------------------------------------------------------------------------------
Residual: (2, 256, 30, 40)

Output
------------------------------------------------------------------------------------------
Dynamic probability: (2, 1, 30, 40)

Range
Min: 0.4373929500579834
Max: 0.9160410165786743

Parameters with gradients: 8

✓ DYNAMIC HEAD TEST PASSED


In [17]:
from pathlib import Path
import numpy as np


sequence = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/left_camera/imo/train/scene13_dyn_test_01_000000"
)


info_file = sequence / "dataset_info.npz"


print(info_file)


data = np.load(
    info_file,
    allow_pickle=True
)


print()
print("="*80)
print("DATASET INFO KEYS")
print("="*80)


for key in data.keys():

    value = data[key]

    print(
        key,
        type(value),
        getattr(value, "shape", None)
    )

/home/ayon/HDD/EventDatasets/EVIMO2_official/left_camera/imo/train/scene13_dyn_test_01_000000/dataset_info.npz

DATASET INFO KEYS
index <class 'numpy.ndarray'> (670,)
discretization <class 'numpy.ndarray'> ()
K <class 'numpy.ndarray'> (3, 3)
D <class 'numpy.ndarray'> (4,)
meta <class 'numpy.ndarray'> ()


In [18]:
frame = voxel_batch.frames[0]

metadata = frame.metadata


print("="*80)
print("METADATA")
print("="*80)


for item in dir(metadata):

    if not item.startswith("_"):

        print(item)

METADATA
camera_distortion
camera_intrinsics
camera_motion
depth
event_sample_indices
events_p
events_t
events_xy
frame_ids
frame_motion
imu_angular_velocity
imu_linear_acceleration
imu_sample_indices
imu_timestamps
local_frame_indices
mask
rgb
sensors
sequence_names
timestamps


In [19]:
# ==========================================================
# POSE HEAD TEST
# ==========================================================

from src.models.world_model.pose_head import PoseHead


print()
print("=" * 90)
print("POSE HEAD TEST")
print("=" * 90)


# ----------------------------------------------------------
# Input
# ----------------------------------------------------------

imu_motion = motion_embeddings[:, -1].detach()


print()

print("Input")
print("-" * 90)

print(
    "Motion embedding:",
    tuple(imu_motion.shape)
)



# ----------------------------------------------------------
# Model
# ----------------------------------------------------------

pose_head = PoseHead(
    input_dim=128,
).to(device)


pose_head.train()



# ----------------------------------------------------------
# Forward
# ----------------------------------------------------------

pose = pose_head(
    imu_motion
)



print()

print("Output")
print("-" * 90)


print(
    "Pose:",
    tuple(pose.shape)
)



# ----------------------------------------------------------
# Check
# ----------------------------------------------------------

assert pose.shape == (
    imu_motion.shape[0],
    6,
)


assert torch.isfinite(
    pose
).all()



# ----------------------------------------------------------
# Statistics
# ----------------------------------------------------------

print()

print("Statistics")

print(
    "Translation:",
    pose[:, :3]
)

print(
    "Rotation:",
    pose[:, 3:]
)



# ----------------------------------------------------------
# Gradient
# ----------------------------------------------------------

loss = pose.mean()


pose_head.zero_grad()


loss.backward()


num_grad = 0


for name, param in pose_head.named_parameters():

    if param.grad is not None:

        assert torch.isfinite(
            param.grad
        ).all()

        num_grad += 1



print()

print(
    "Parameters with gradients:",
    num_grad
)


assert num_grad > 0



print()

print("=" * 90)
print("✓ POSE HEAD TEST PASSED")
print("=" * 90)


POSE HEAD TEST

Input
------------------------------------------------------------------------------------------
Motion embedding: (2, 128)

Output
------------------------------------------------------------------------------------------
Pose: (2, 6)

Statistics
Translation: tensor([[ 0.4204, -0.3022,  0.2148],
        [ 0.3804, -0.2868,  0.1446]], grad_fn=<SliceBackward0>)
Rotation: tensor([[-0.0929, -0.0277,  0.2487],
        [-0.0016,  0.1104,  0.2248]], grad_fn=<SliceBackward0>)

Parameters with gradients: 10

✓ POSE HEAD TEST PASSED


In [20]:
# ==========================================================
# DEPTH HEAD TEST
# ==========================================================

from src.models.world_model.depth_head import DepthHead


print()
print("=" * 90)
print("DEPTH HEAD TEST")
print("=" * 90)



# ----------------------------------------------------------
# Input
# ----------------------------------------------------------

latent_feature = temporal_features[:, -1].detach()


print()

print("Input")
print("-" * 90)

print(
    "Feature:",
    tuple(latent_feature.shape)
)



# ----------------------------------------------------------
# Model
# ----------------------------------------------------------

depth_head = DepthHead(
    input_channels=256,
).to(device)


depth_head.train()



# ----------------------------------------------------------
# Forward
# ----------------------------------------------------------

depth = depth_head(
    latent_feature
)



print()

print("Output")
print("-" * 90)

print(
    "Depth:",
    tuple(depth.shape)
)



# ----------------------------------------------------------
# Assertions
# ----------------------------------------------------------

assert depth.shape == (
    latent_feature.shape[0],
    1,
    latent_feature.shape[2],
    latent_feature.shape[3],
)


assert torch.isfinite(
    depth
).all()


assert torch.all(
    depth > 0
)



# ----------------------------------------------------------
# Statistics
# ----------------------------------------------------------

print()

print("Statistics")

print(
    "Min:",
    depth.min().item()
)

print(
    "Max:",
    depth.max().item()
)

print(
    "Mean:",
    depth.mean().item()
)

print(
    "Std:",
    depth.std().item()
)



# ----------------------------------------------------------
# Gradient
# ----------------------------------------------------------

loss = depth.mean()


depth_head.zero_grad()


loss.backward()



num_grad = 0


for name, param in depth_head.named_parameters():

    if param.grad is not None:

        assert torch.isfinite(
            param.grad
        ).all()

        num_grad += 1



print()

print(
    "Parameters with gradients:",
    num_grad
)


assert num_grad > 0



print()

print("=" * 90)
print("✓ DEPTH HEAD TEST PASSED")
print("=" * 90)


DEPTH HEAD TEST

Input
------------------------------------------------------------------------------------------
Feature: (2, 256, 30, 40)

Output
------------------------------------------------------------------------------------------
Depth: (2, 1, 30, 40)

Statistics
Min: 0.2058240920305252
Max: 2.2149815559387207
Mean: 0.770738959312439
Std: 0.1645858734846115

Parameters with gradients: 10

✓ DEPTH HEAD TEST PASSED


In [23]:
# ==========================================================
# LATENT RENDERER TEST (REAL EVIMO2 DATA)
# ==========================================================

import torch

from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.models.world_model.latent_renderer import LatentRenderer

print("=" * 90)
print("LATENT RENDERER (REAL DATA)")
print("=" * 90)

# ----------------------------------------------------------
# Dataset
# ----------------------------------------------------------



dataset_root = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

frame_dataset = EVIMO2Dataset(
    dataset_root=dataset_root,
    sensors=("left_camera", "right_camera"),
    split="train",
    load_depth=True,
    load_mask=True,
)

dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=(-3, -2, -1, 0),
)


batch = temporal_collate_fn(
    [
        dataset[0],
        dataset[1],
    ]
)

#
# Use the newest frame
#
frame = batch.frames[-1]

# ----------------------------------------------------------
# Real camera calibration
# ----------------------------------------------------------

K = torch.from_numpy(
    __import__("numpy").stack(frame.camera_intrinsics)
).float()

distortion = torch.from_numpy(
    __import__("numpy").stack(frame.camera_distortion)
).float()

# ----------------------------------------------------------
# Dummy latent feature
# (replace later by World Transition output)
# ----------------------------------------------------------

B = len(frame.camera_intrinsics)

feature = torch.randn(
    B,
    256,
    30,
    40,
    requires_grad=True,
)

# ----------------------------------------------------------
# Dummy predicted depth
# (replace later by DepthHead)
# ----------------------------------------------------------

depth = torch.rand(
    B,
    1,
    30,
    40,
)

depth = depth + 1.0

# ----------------------------------------------------------
# Dummy predicted pose
# (replace later by PoseHead)
# ----------------------------------------------------------

pose = torch.zeros(
    B,
    6,
)

pose[:,0] = 0.02
pose[:,5] = 0.01

# ----------------------------------------------------------
# Renderer
# ----------------------------------------------------------

renderer = LatentRenderer()

warped = renderer(
    feature,
    depth,
    pose,
    K,
    distortion,
)

# ----------------------------------------------------------
# Gradient test
# ----------------------------------------------------------

loss = warped.mean()

loss.backward()

# ----------------------------------------------------------
# Statistics
# ----------------------------------------------------------

print()
print("Inputs")
print("-"*90)

print("Feature      :", tuple(feature.shape))
print("Depth        :", tuple(depth.shape))
print("Pose         :", tuple(pose.shape))
print("K            :", tuple(K.shape))
print("Distortion   :", tuple(distortion.shape))

print()
print("Output")
print("-"*90)

print("Warped       :", tuple(warped.shape))

print()
print("Statistics")
print("-"*90)

print("Mean         :", warped.mean().item())
print("Std          :", warped.std().item())

print("NaNs         :", torch.isnan(warped).any().item())
print("Infs         :", torch.isinf(warped).any().item())

print()

print("Gradient OK  :", feature.grad is not None)

print()
print("=" * 90)
print("✓ LATENT RENDERER REAL-DATA TEST PASSED")
print("=" * 90)

LATENT RENDERER (REAL DATA)
EVIMO2 Sequence Index
Sequences : 22
Frames    : 9352
Sensors   : left_camera, right_camera
Split     : train

Inputs
------------------------------------------------------------------------------------------
Feature      : (2, 256, 30, 40)
Depth        : (2, 1, 30, 40)
Pose         : (2, 6)
K            : (2, 3, 3)
Distortion   : (2, 4)

Output
------------------------------------------------------------------------------------------
Warped       : (2, 256, 30, 40)

Statistics
------------------------------------------------------------------------------------------
Mean         : 0.0009888967033475637
Std          : 0.7174673676490784
NaNs         : False
Infs         : False

Gradient OK  : True

✓ LATENT RENDERER REAL-DATA TEST PASSED


In [24]:
import torch

from src.models.world_model.alignment import Alignment

print("="*90)
print("ALIGNMENT TEST")
print("="*90)

model = Alignment(
    channels=256,
)

x = torch.randn(
    2,
    4,
    256,
    30,
    40,
    requires_grad=True,
)

print("\nInput")
print("-"*90)
print(x.shape)

y = model(x)

print("\nOutput")
print("-"*90)
print(y.shape)

print("\nStatistics")
print("-"*90)
print("Mean :", y.mean().item())
print("Std  :", y.std().item())
print("NaNs :", torch.isnan(y).any().item())
print("Infs :", torch.isinf(y).any().item())

loss = y.mean()
loss.backward()

print("\nGradient")
print("-"*90)
print("Input grad :", x.grad is not None)

print("\n" + "="*90)
print("✓ ALIGNMENT TEST PASSED")
print("="*90)

ALIGNMENT TEST

Input
------------------------------------------------------------------------------------------
torch.Size([2, 4, 256, 30, 40])

Output
------------------------------------------------------------------------------------------
torch.Size([2, 4, 256, 30, 40])

Statistics
------------------------------------------------------------------------------------------
Mean : 0.563782811164856
Std  : 0.8267780542373657
NaNs : False
Infs : False

Gradient
------------------------------------------------------------------------------------------
Input grad : True

✓ ALIGNMENT TEST PASSED


In [25]:
import torch

from src.models.world_model.temporal_memory import TemporalMemory

print("="*90)
print("TEMPORAL MEMORY TEST")
print("="*90)

model = TemporalMemory()

x = torch.randn(
    2,
    4,
    256,
    30,
    40,
    requires_grad=True,
)

print("\nInput")
print("-"*90)
print(x.shape)

y = model(x)

print("\nOutput")
print("-"*90)
print(y.shape)

print("\nStatistics")
print("-"*90)
print("Mean :", y.mean().item())
print("Std  :", y.std().item())
print("NaNs :", torch.isnan(y).any().item())
print("Infs :", torch.isinf(y).any().item())

loss = y.mean()
loss.backward()

print("\nGradient")
print("-"*90)
print("Input grad :", x.grad is not None)

print("\n" + "="*90)
print("✓ TEMPORAL MEMORY TEST PASSED")
print("="*90)

TEMPORAL MEMORY TEST

Input
------------------------------------------------------------------------------------------
torch.Size([2, 4, 256, 30, 40])

Output
------------------------------------------------------------------------------------------
torch.Size([2, 256, 30, 40])

Statistics
------------------------------------------------------------------------------------------
Mean : -8.542459517535406e-10
Std  : 0.999996542930603
NaNs : False
Infs : False

Gradient
------------------------------------------------------------------------------------------
Input grad : True

✓ TEMPORAL MEMORY TEST PASSED


In [4]:
import torch

from src.models.world_model.decoder import WorldDecoder

print("=" * 90)
print("WORLD DECODER TEST")
print("=" * 90)

model = WorldDecoder()

x = torch.randn(
    2,
    256,
    30,
    40,
    requires_grad=True,
)

print("\nInput")
print("-" * 90)
print(x.shape)

y = model(x)

print("\nOutput")
print("-" * 90)
print(y.shape)

print("\nStatistics")
print("-" * 90)
print("Mean :", y.mean().item())
print("Std  :", y.std().item())
print("NaNs :", torch.isnan(y).any().item())
print("Infs :", torch.isinf(y).any().item())

loss = y.mean()
loss.backward()

print("\nGradient")
print("-" * 90)
print("Input grad :", x.grad is not None)

print("\n" + "=" * 90)
print("✓ WORLD DECODER TEST PASSED")
print("=" * 90)

WORLD DECODER TEST

Input
------------------------------------------------------------------------------------------
torch.Size([2, 256, 30, 40])
jij

Output
------------------------------------------------------------------------------------------
torch.Size([2, 16, 480, 640])

Statistics
------------------------------------------------------------------------------------------
Mean : 0.2562248706817627
Std  : 0.6196495890617371
NaNs : False
Infs : False

Gradient
------------------------------------------------------------------------------------------
Input grad : True

✓ WORLD DECODER TEST PASSED


In [5]:
import torch

from src.models.world_model.mask_head import DynamicMaskHead

print("=" * 90)
print("DYNAMIC MASK HEAD TEST")
print("=" * 90)

model = DynamicMaskHead()

x = torch.randn(
    2,
    16,
    480,
    640,
    requires_grad=True,
)

print("\nInput")
print("-" * 90)
print(x.shape)

y = model(x)

print("\nOutput")
print("-" * 90)
print(y.shape)

print("\nStatistics")
print("-" * 90)
print("Min  :", y.min().item())
print("Max  :", y.max().item())
print("Mean :", y.mean().item())
print("Std  :", y.std().item())

print("NaNs :", torch.isnan(y).any().item())
print("Infs :", torch.isinf(y).any().item())

loss = y.mean()
loss.backward()

print("\nGradient")
print("-" * 90)
print("Input grad :", x.grad is not None)

print("\n" + "=" * 90)
print("✓ DYNAMIC MASK HEAD TEST PASSED")
print("=" * 90)

DYNAMIC MASK HEAD TEST

Input
------------------------------------------------------------------------------------------
torch.Size([2, 16, 480, 640])

Output
------------------------------------------------------------------------------------------
torch.Size([2, 1, 480, 640])

Statistics
------------------------------------------------------------------------------------------
Min  : 0.14768299460411072
Max  : 0.7380891442298889
Mean : 0.44118455052375793
Std  : 0.06290580332279205
NaNs : False
Infs : False

Gradient
------------------------------------------------------------------------------------------
Input grad : True

✓ DYNAMIC MASK HEAD TEST PASSED


In [ ]:
# ==========================================================
# END-TO-END WORLD MODEL FORWARD PASS
# ==========================================================

import torch

from src.models.world_model.event_encoder import EventEncoder
from src.models.world_model.temporal_encoder import TemporalEncoder
from src.models.world_model.depth_head import DepthHead
from src.models.world_model.imu_encoder import IMUEncoder
from src.models.world_model.pose_head import PoseHead
from src.models.world_model.latent_renderer import LatentRenderer
from src.models.world_model.fusion import MotionFusion
from src.models.world_model.alignment import Alignment
from src.models.world_model.temporal_memory import TemporalMemory
from src.models.world_model.decoder import WorldDecoder
from src.models.world_model.mask_head import DynamicMaskHead

print("=" * 90)
print("END-TO-END WORLD MODEL")
print("=" * 90)

# ----------------------------------------------------------
# Build models
# ----------------------------------------------------------

event_encoder = EventEncoder(
    input_channels=num_bins,
).to(device)

temporal_encoder = TemporalEncoder(
    input_channels=256,
    hidden_channels=256,
).to(device)

depth_head = DepthHead(
    input_channels=256,
).to(device)

imu_encoder = IMUEncoder(
    hidden_channels=64,
    embedding_dim=128,
).to(device)

pose_head = PoseHead(
    input_dim=128,
).to(device)

renderer = LatentRenderer().to(device)

fusion = MotionFusion(
    event_channels=256,
    imu_dim=128,
).to(device)

alignment = Alignment(
    channels=256,
).to(device)

memory = TemporalMemory().to(device)

decoder = WorldDecoder().to(device)

mask_head = DynamicMaskHead().to(device)

# ----------------------------------------------------------
# Event Encoder
# ----------------------------------------------------------

feature_pyramid = event_encoder(voxels)

latent = feature_pyramid[-1]

print()
print("Latent feature :", tuple(latent.shape))

# ----------------------------------------------------------
# Temporal Encoder
# ----------------------------------------------------------

temporal_features = temporal_encoder(
    latent
)

print("Temporal feature :", tuple(temporal_features.shape))

# ----------------------------------------------------------
# Depth Prediction
# ----------------------------------------------------------

reference_feature = temporal_features[:, -1]

depth = depth_head(
    reference_feature
)

print("Depth :", tuple(depth.shape))

# ----------------------------------------------------------
# IMU Encoder
# ----------------------------------------------------------

motion_embeddings = []

for frame in voxel_batch.frames:

    embedding = imu_encoder(
        frame=frame,
        batch_size=batch_size,
    )

    motion_embeddings.append(
        embedding
    )

motion_embeddings = torch.stack(
    motion_embeddings,
    dim=1,
)

print("Motion embedding :", tuple(motion_embeddings.shape))

# ----------------------------------------------------------
# Pose Prediction
# ----------------------------------------------------------

pose = pose_head(
    motion_embeddings[:, -1]
)

print("Pose :", tuple(pose.shape))

# ----------------------------------------------------------
# Camera Calibration
# ----------------------------------------------------------

reference_batch = voxel_batch.frames[-1]

K = torch.stack(
    reference_batch.camera_intrinsics
).to(device)

distortion = torch.stack(
    reference_batch.camera_distortion
).to(device)

print("K :", tuple(K.shape))
print("Distortion :", tuple(distortion.shape))

# ----------------------------------------------------------
# Latent Rendering
# ----------------------------------------------------------

warped = renderer(
    feature=reference_feature,
    depth=depth,
    pose=pose,
    K=K,
    distortion=distortion,
)

print("Warped :", tuple(warped.shape))

# ----------------------------------------------------------
# Motion Fusion
# ----------------------------------------------------------

warped_sequence = warped.unsqueeze(1).repeat(
    1,
    sequence_length,
    1,
    1,
    1,
)

fused = fusion(
    warped_sequence,
    motion_embeddings,
)

print("Fusion :", tuple(fused.shape))

# ----------------------------------------------------------
# Alignment
# ----------------------------------------------------------

aligned = alignment(
    fused
)

print("Aligned :", tuple(aligned.shape))

# ----------------------------------------------------------
# Temporal Memory
# ----------------------------------------------------------

world_state = memory(
    aligned
)

print("World state :", tuple(world_state.shape))

# ----------------------------------------------------------
# Decoder
# ----------------------------------------------------------

decoded = decoder(
    world_state
)

print("Decoded :", tuple(decoded.shape))

# ----------------------------------------------------------
# Dynamic Mask
# ----------------------------------------------------------

mask = mask_head(
    decoded
)

print("Mask :", tuple(mask.shape))

# ----------------------------------------------------------
# Checks
# ----------------------------------------------------------

assert torch.isfinite(mask).all()

print()
print("-" * 90)
print("Statistics")
print("-" * 90)

print("Min :", mask.min().item())
print("Max :", mask.max().item())
print("Mean:", mask.mean().item())
print("Std :", mask.std().item())

# ----------------------------------------------------------
# Gradient
# ----------------------------------------------------------

loss = mask.mean()

for model in [
    event_encoder,
    temporal_encoder,
    depth_head,
    imu_encoder,
    pose_head,
    renderer,
    fusion,
    alignment,
    memory,
    decoder,
    mask_head,
]:
    model.zero_grad()

loss.backward()

num_grad = 0

for module in [
    event_encoder,
    temporal_encoder,
    depth_head,
    imu_encoder,
    pose_head,
    fusion,
    alignment,
    memory,
    decoder,
    mask_head,
]:

    for p in module.parameters():

        if p.grad is not None:

            assert torch.isfinite(p.grad).all()
            num_grad += 1

print()
print("-" * 90)
print("Gradient")
print("-" * 90)

print("Parameters with gradients :", num_grad)

assert num_grad > 0

if torch.cuda.is_available():

    print()
    print("-" * 90)
    print("CUDA")
    print("-" * 90)

    print(
        f"Allocated : {torch.cuda.memory_allocated()/1024**2:.2f} MB"
    )

    print(
        f"Reserved  : {torch.cuda.memory_reserved()/1024**2:.2f} MB"
    )

print()
print("=" * 90)
print("✓ END-TO-END WORLD MODEL FORWARD PASSED")
print("=" * 90)